# Generate masks from GeoJSON
Classes: 0 land, 1 road, 2 building, 255 no-data (ignore). Settings live in `src/config.py`.

Pipeline: `coverage.py` (road-label coverage per tile) -> `build_dataset.py` (filter, images, masks, tiles.csv, overlays) -> `check_masks.py` (overlay checks) -> `split.py` (train/val/test).

In [ ]:
import sys; sys.path.insert(0, '../src')
import config as C
print(C.CLASS_NAMES, C.METRIC_CRS, C.LANE_WIDTH_M, C.MIN_ROAD_COVERAGE, C.MAX_NODATA)

## 1. Road-label coverage (SN3 footprints vs SN2 tiles)

In [ ]:
import coverage
cov = coverage.road_coverage()
print((cov['road_coverage_%'] >= C.MIN_ROAD_COVERAGE).sum(), 'of', len(cov), 'tiles >=', C.MIN_ROAD_COVERAGE, '%')

## 2. Build dataset: filter tiles, write images/masks, tiles.csv, overlays
Runs coverage.py itself; clears and rewrites data/images, data/masks, results/mask_check.

In [ ]:
%run ../src/build_dataset.py

## 3. Overlay checks (re-run only)

In [ ]:
import check_masks
check_masks.run()
check_masks.run_no_roads()

## 4. Train/val/test split (70/15/15, seed 42)

In [ ]:
import split
s = split.make_splits()
for name, g in s.groupby('split'):
    print(name, len(g), split.class_percentages(g.tile_id))